# Scorecard Refresh — tier & health analysis

An **analysis artifact**, not an agent skill. Recomputing partner tiers and
health flags is a periodic close-time analysis a human runs and eyeballs — it
doesn't belong in `.claude/skills/` next to the recurring agent workflows.

It recomputes the Partner Value Score, tier, and health flag for every partner
straight from the thresholds in [`data/DATA_DICTIONARY.md`](../data/DATA_DICTIONARY.md),
then diffs the result against the committed `partner_metrics.csv`. The dictionary
is the source of truth; this notebook computes, it never decides.

In [ ]:
from datetime import date
import pandas as pd

TODAY = date(2026, 6, 13)
df = pd.read_csv("../data/partner_metrics.csv")
df.head()

## Partner Value Score (caps per the data dictionary)

Revenue 40 · Deal registration 20 · Technical capacity 20 · Satisfaction 20 —
each component capped so no single dimension can buy a tier.

In [ ]:
def value_score(r):
    revenue = min(r.attributed_revenue_fy26 / 2_500_000, 1) * 40
    dealreg = min(r.deal_regs_approved_fy26 / 40, 1) * 20
    technical = min(r.certified_engineers / 40, 1) * 20
    satisfaction = max(r.nps, 0) / 100 * 20
    return round(revenue + dealreg + technical + satisfaction)

def tier(score):
    if score >= 70:
        return "Strategic"
    if score >= 40:
        return "Premier"
    return "Select"

def health(r):
    stale = (TODAY - date.fromisoformat(r.last_qbr_date)).days > 120
    if r.nps < 50 and stale:
        return "red"
    if r.nps < 55 or stale:
        return "yellow"
    return "green"

df["pvs_recomputed"] = df.apply(value_score, axis=1)
df["tier_recomputed"] = df["pvs_recomputed"].apply(tier)
df["health_recomputed"] = df.apply(health, axis=1)

## Deltas only

Show just the partners whose tier or health flag would move — the same
delta-before-write discipline the old skill enforced. Partners within ±3 points
of a tier threshold are flagged for review rather than moved automatically.

In [ ]:
moved = df[(df.tier != df.tier_recomputed) | (df.health_flag != df.health_recomputed)]
cols = ["partner_name", "tier", "tier_recomputed", "health_flag", "health_recomputed", "pvs_recomputed"]
print("Tier/health changes:" if len(moved) else "No tier or health changes — scorecard is current.")
moved[cols]

In [ ]:
border = df[(df.pvs_recomputed - 70).abs().le(3) | (df.pvs_recomputed - 40).abs().le(3)]
print("Within ±3 of a tier threshold — flag for review, don't auto-move:")
border[["partner_name", "pvs_recomputed", "tier_recomputed"]]